# Task 3: Image Enhancement Using Spatial and Frequency Domain Techniques

This notebook solves Task 3 of the homework. All core algorithms are implemented manually:

- spatial smoothing, high-pass filtering, and unsharp masking use a custom 2D convolution function;
- the naive DFT evaluates the mathematical DFT formula directly and does not use `np.fft`;
- the FFT/IFFT are custom recursive Cooley-Tukey implementations extended to 2D by separability;
- frequency masks, reconstruction, magnitude/phase experiments, and energy metrics are computed from the custom Fourier transform.

The main test image is `images/Task_3_Image_Enhancement/pnois1.jpg` because it is already grayscale and has size `512 x 512`, which is suitable for radix-2 FFT. The other Task 3 images are also loaded for reference.

## 1. Setup and Imports

This cell imports only general-purpose packages. The assignment restriction forbids built-in image-processing and Fourier-transform functions such as `cv2.filter2D`, `scipy.signal.convolve`, and `np.fft.*`; none of those are used here.

Images are represented as floating-point arrays in the range `[0, 255]`.

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
import time

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

IMAGE_DIR = Path("images/Task_3_Image_Enhancement")
OUTPUT_DIR = Path("task3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["image.cmap"] = "gray"


def clip_uint8(image):
    'Clip a floating image to displayable 8-bit range.'
    return np.clip(image, 0, 255)


def normalize_for_display(image):
    'Linearly normalize an array to [0, 255] for visualization only.'
    image = np.asarray(image, dtype=np.float64)
    mn, mx = np.min(image), np.max(image)
    if mx - mn < 1e-12:
        return np.zeros_like(image)
    return 255.0 * (image - mn) / (mx - mn)


def show_images(images, titles, cols=3, figsize=(14, 8), cmap="gray"):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image, cmap=cmap, vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[len(images):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 2. Load Task 3 Images

The images are converted to grayscale using the luminance formula

$$
I = 0.299R + 0.587G + 0.114B
$$

For the frequency-domain work we use a square power-of-two image, because the radix-2 FFT recursively splits the signal into even and odd samples.

In [ ]:
def read_grayscale(path):
    img = Image.open(path)
    arr = np.asarray(img, dtype=np.float64)
    if arr.ndim == 2:
        return arr
    return 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]


task3_paths = sorted(IMAGE_DIR.glob("*"))
loaded_images = {path.name: read_grayscale(path) for path in task3_paths}

# print(loaded_images)

for name, arr in loaded_images.items():
    print(f"{name:12s} shape={arr.shape}, min={arr.min():.1f}, max={arr.max():.1f}")

base_image = loaded_images["pnois1.jpg"]
show_images(
    [clip_uint8(img) for img in loaded_images.values()],
    list(loaded_images.keys()),
    cols=3,
    figsize=(13, 5),
)

## 3. Manual Padding, Convolution, and Gaussian Kernel

For an image `f` and a convolution kernel `h` with size `(2a + 1) x (2b + 1)`, each output pixel is computed as:

$$
g(x,y) = \sum_{i=-a}^{a} \sum_{j=-b}^{b} f(x-i, y-j) \cdot h(i,j)
$$

The notebook implements two boundary modes required by the assignment:

- **zero padding:** missing pixels outside the image are treated as zero;
- **mirror padding / edge replication:** the outermost border pixels are duplicated outward. This avoids strong artificial black borders during smoothing and sharpening.

In [ ]:
def pad_image(image, pad_y, pad_x, mode="mirror"):
    image = np.asarray(image, dtype=np.float64)
    if mode == "zero":
        return np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), mode="constant", constant_values=0)
    if mode == "mirror":
        return np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), mode="edge")
    raise ValueError("mode must be 'zero' or 'mirror'")


def convolve2d(image, kernel, padding="mirror"):
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)
    kh, kw = kernel.shape
    py, px = kh // 2, kw // 2
    padded = pad_image(image, py, px, mode=padding)
    flipped = kernel[::-1, ::-1]
    out = np.zeros_like(image, dtype=np.float64)
    
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            region = padded[y:y + kh, x:x + kw]
            out[y, x] = np.sum(region * flipped)
    return out


def gaussian_kernel(size=7, sigma=1.4):
    if size % 2 == 0:
        raise ValueError("Gaussian kernel size must be odd")
    radius = size // 2
    y, x = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    return kernel / np.sum(kernel)


G7 = gaussian_kernel(size=7, sigma=1.4)
print("7x7 Gaussian kernel, sigma=1.4")
print(np.round(G7, 4))

## 4. Spatial Domain Sharpening

Unsharp masking first extracts a high-frequency mask by subtracting a smoothed image:

$$
m(x,y)=f(x,y)-\bar{f}(x,y)
$$

Then it adds a scaled version of this mask back to the original image:

$$
g(x,y)=f(x,y)+k\left(f(x,y)-\bar{f}(x,y)\right)
$$

This cell also applies a direct spatial high-pass kernel:

$$
H=
\begin{bmatrix}
-1 & -1 & -1\\
-1 & 8 & -1\\
-1 & -1 & -1
\end{bmatrix}
$$

Mirror padding is used here because replicated borders avoid the artificial dark frame that zero padding can introduce in sharpening results.

In [ ]:
smooth = convolve2d(base_image, gaussian_kernel(size=7, sigma=1.4), padding="mirror")
high_frequency_mask = base_image - smooth

k = 1.3
unsharp = clip_uint8(base_image + k * high_frequency_mask)

high_pass_kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1],
], dtype=np.float64)

spatial_high_pass_response = convolve2d(base_image, high_pass_kernel, padding="mirror")
spatial_high_pass_display = normalize_for_display(np.abs(spatial_high_pass_response))
spatial_high_pass_enhanced = clip_uint8(base_image + 0.30 * spatial_high_pass_response)

show_images(
    [
        clip_uint8(base_image),
        clip_uint8(smooth),
        normalize_for_display(high_frequency_mask),
        unsharp,
        spatial_high_pass_display,
        spatial_high_pass_enhanced,
    ],
    [
        "Original grayscale",
        "Gaussian-smoothed image",
        "Unsharp mask: f - f_bar",
        "Spatially sharpened image",
        "High-pass response |H * f|",
        "High-pass enhanced image",
    ],
    cols=3,
    figsize=(14, 8),
)

## 5. Manual DFT, FFT, IFFT, and Frequency Centering

The 2D DFT is

$$
F(u,v)=\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

The naive implementation below evaluates every coefficient directly from this formula and therefore has \(O(N^4)\) cost for an \(N\times N\) image.

The FFT implementation is a recursive radix-2 Cooley-Tukey transform:

$$
X_k=E_k+e^{-j2\pi k/N}O_k,\qquad
X_{k+N/2}=E_k-e^{-j2\pi k/N}O_k
$$

The 2D FFT uses separability: apply the 1D FFT to every row, then to every column. IFFT uses the conjugate identity

$$
\operatorname{IFFT}(X)=\frac{1}{N}\overline{\operatorname{FFT}(\overline{X})}
$$

To center the DC component, the image is multiplied by `(-1.0) ** (x + y)` before the transform.

In [ ]:
def checkerboard(shape):
    y, x = np.indices(shape)
    return (-1.0) ** (x + y)


def dft2_naive_formula(image, vectorized_inner=True):
    image = np.asarray(image, dtype=np.complex128)
    rows, cols = image.shape
    F = np.zeros((rows, cols), dtype=np.complex128)
    
    if vectorized_inner:
        x = np.arange(rows)[:, None]
        y = np.arange(cols)[None, :]
        for u in range(rows):
            for v in range(cols):
                exponent = -2j * np.pi * ((u * x / rows) + (v * y / cols))
                F[u, v] = np.sum(image * np.exp(exponent))
        return F
    
    for u in range(rows):
        for v in range(cols):
            total = 0.0j
            for x in range(rows):
                for y in range(cols):
                    exponent = -2j * np.pi * ((u * x / rows) + (v * y / cols))
                    total += image[x, y] * np.exp(exponent)
            F[u, v] = total
    return F


def fft1d(signal):
    signal = np.asarray(signal, dtype=np.complex128)
    n = signal.shape[0]
    if n == 1:
        return signal.copy()
    if n % 2 != 0:
        raise ValueError("FFT length must be a power of two")
    
    even = fft1d(signal[::2])
    odd = fft1d(signal[1::2])
    twiddle = np.exp(-2j * np.pi * np.arange(n // 2) / n)
    first = even + twiddle * odd
    second = even - twiddle * odd
    return np.concatenate([first, second])


def ifft1d(spectrum):
    spectrum = np.asarray(spectrum, dtype=np.complex128)
    n = spectrum.shape[0]
    return np.conjugate(fft1d(np.conjugate(spectrum))) / n


def fft2(image):
    image = np.asarray(image, dtype=np.complex128)
    row_fft = np.zeros_like(image, dtype=np.complex128)
    for r in range(image.shape[0]):
        row_fft[r, :] = fft1d(image[r, :])
    
    out = np.zeros_like(row_fft, dtype=np.complex128)
    for c in range(image.shape[1]):
        out[:, c] = fft1d(row_fft[:, c])
    return out


def ifft2(spectrum):
    spectrum = np.asarray(spectrum, dtype=np.complex128)
    row_ifft = np.zeros_like(spectrum, dtype=np.complex128)
    for r in range(spectrum.shape[0]):
        row_ifft[r, :] = ifft1d(spectrum[r, :])
    
    out = np.zeros_like(row_ifft, dtype=np.complex128)
    for c in range(spectrum.shape[1]):
        out[:, c] = ifft1d(row_ifft[:, c])
    return out


def crop_center_square(image, size):
    h, w = image.shape
    y0 = (h - size) // 2
    x0 = (w - size) // 2
    return image[y0:y0 + size, x0:x0 + size]


def resize_to_square(image, size):
    img = Image.fromarray(clip_uint8(image).astype(np.uint8))
    return np.asarray(img.resize((size, size), Image.Resampling.BILINEAR), dtype=np.float64)


# Quick correctness check on a small image: DFT and FFT should agree numerically.
small = resize_to_square(base_image, 8)
small_centered = small * checkerboard(small.shape)
F_dft_small = dft2_naive_formula(small_centered, vectorized_inner=False)
F_fft_small = fft2(small_centered)
print("max |DFT - FFT| on 8x8 check:", np.max(np.abs(F_dft_small - F_fft_small)))

## 6. Benchmark Naive DFT vs. Custom 2D FFT

The assignment asks for runtime measurements at \(N=32,64,128\). The naive DFT computes each frequency coefficient independently from the DFT formula, while the FFT reuses subproblem results. The expected complexity difference is:

$$
\text{Naive 2D DFT}: O(N^4),\qquad \text{2D FFT}: O(N^2\log N)
$$

The function `dft2_naive_formula` contains a fully nested-loop implementation (`vectorized_inner=False`). The benchmark below runs that pure four-loop version for `N=32` as a correctness/timing check, then uses the same direct DFT formula with a vectorized inner summation for the required `N=32,64,128` timings so the notebook remains practical. No `np.fft` or built-in Fourier transform is used.


In [ ]:
benchmark_sizes = [32, 64, 128]
benchmark_rows = []
pure_loop_reference = None

for N in benchmark_sizes:
    patch = resize_to_square(base_image, N)
    centered_patch = patch * checkerboard(patch.shape)
    
    # Pure nested-loop DFT is very slow at larger N, so run it once at N=32
    # to document that the direct four-loop implementation is functional.
    if N == 32:
        t0 = time.perf_counter()
        F_pure_loop = dft2_naive_formula(centered_patch, vectorized_inner=False)
        pure_loop_time = time.perf_counter() - t0
        pure_loop_reference = F_pure_loop
    else:
        pure_loop_time = None
    
    t0 = time.perf_counter()
    F_formula = dft2_naive_formula(centered_patch, vectorized_inner=True)
    naive_time = time.perf_counter() - t0
    
    if pure_loop_reference is not None and N == 32:
        max_diff = np.max(np.abs(pure_loop_reference - F_formula))
        print(f"Pure nested-loop DFT check at N=32: {pure_loop_time:.4f}s, max |pure - vectorized| = {max_diff:.3e}")
    
    t0 = time.perf_counter()
    _ = fft2(centered_patch)
    fft_time = time.perf_counter() - t0
    
    benchmark_rows.append((N, naive_time, fft_time, pure_loop_time, naive_time / max(fft_time, 1e-12)))
    print(f"N={N:3d} | direct DFT formula={naive_time:9.4f}s | FFT={fft_time:9.4f}s | speedup={naive_time / max(fft_time, 1e-12):8.2f}x")

sizes = [row[0] for row in benchmark_rows]
naive_times = [row[1] for row in benchmark_rows]
fft_times = [row[2] for row in benchmark_rows]

plt.figure(figsize=(8, 5))
plt.plot(sizes, naive_times, marker="o", label="Direct DFT formula")
plt.plot(sizes, fft_times, marker="o", label="Custom 2D FFT")
if benchmark_rows[0][3] is not None:
    plt.scatter([benchmark_rows[0][0]], [benchmark_rows[0][3]], marker="x", s=90, label="Pure four-loop DFT check")
plt.xlabel("Square image size N")
plt.ylabel("Execution time (seconds)")
plt.title("Naive DFT vs. Custom FFT Runtime")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 7. Frequency Representation: Magnitude and Phase

The Fourier transform is complex:

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}
$$

where

$$
|F(u,v)|=\sqrt{\operatorname{Re}(F)^2+\operatorname{Im}(F)^2},\qquad
\phi(u,v)=\tan^{-1}\left(\frac{\operatorname{Im}(F)}{\operatorname{Re}(F)}\right)
$$

The magnitude spectrum is displayed using logarithmic compression:

$$
S(u,v)=\log(1+|F(u,v)|)
$$

This makes weak high-frequency details visible beside the very large DC/low-frequency values.

In [ ]:
freq_image = crop_center_square(base_image, 512)
centered_image = freq_image * checkerboard(freq_image.shape)

F = fft2(centered_image)
magnitude = np.abs(F)
phase = np.angle(F)
log_magnitude = normalize_for_display(np.log1p(magnitude))
phase_display = normalize_for_display(phase)

show_images(
    [clip_uint8(freq_image), log_magnitude, phase_display],
    ["Original grayscale image", "Log-scaled magnitude spectrum", "Phase angle map"],
    cols=3,
    figsize=(14, 5),
)

## 8. Frequency-Domain Low-Pass and High-Pass Filtering

Let \(D(u,v)\) be the distance from the centered frequency coordinate \((u,v)\) to the origin. The filters used here are:

Ideal low-pass:

$$
H_{ILP}(u,v)=
\begin{cases}
1, & D(u,v)\le D_0\\
0, & D(u,v)>D_0
\end{cases}
$$

Gaussian low-pass:

$$
H_{GLP}(u,v)=e^{-D(u,v)^2/(2D_0^2)}
$$

High-pass filtering is implemented as the complement:

$$
H_{HP}(u,v)=1-H_{LP}(u,v)
$$

After filtering, reconstruction is done with the custom IFFT and the centering factor \((-1)^{x+y}\) is removed.

In [ ]:
def centered_distance_grid(shape):
    rows, cols = shape
    y, x = np.indices(shape)
    cy, cx = rows // 2, cols // 2
    return np.sqrt((y - cy) ** 2 + (x - cx) ** 2)


def ideal_low_pass(shape, cutoff):
    D = centered_distance_grid(shape)
    return (D <= cutoff).astype(np.float64)


def gaussian_low_pass(shape, cutoff):
    D = centered_distance_grid(shape)
    return np.exp(-(D ** 2) / (2 * cutoff ** 2))


def reconstruct_from_centered_spectrum(spectrum):
    centered_reconstruction = ifft2(spectrum).real
    return centered_reconstruction * checkerboard(centered_reconstruction.shape)


cutoff = 45
H_ideal_lp = ideal_low_pass(freq_image.shape, cutoff)
H_gaussian_lp = gaussian_low_pass(freq_image.shape, cutoff)
H_ideal_hp = 1.0 - H_ideal_lp
H_gaussian_hp = 1.0 - H_gaussian_lp

ideal_lp_recon = clip_uint8(reconstruct_from_centered_spectrum(F * H_ideal_lp))
gaussian_lp_recon = clip_uint8(reconstruct_from_centered_spectrum(F * H_gaussian_lp))
ideal_hp_recon_raw = reconstruct_from_centered_spectrum(F * H_ideal_hp)
gaussian_hp_recon_raw = reconstruct_from_centered_spectrum(F * H_gaussian_hp)

ideal_hp_recon = normalize_for_display(ideal_hp_recon_raw)
gaussian_hp_recon = normalize_for_display(gaussian_hp_recon_raw)

show_images(
    [
        normalize_for_display(H_ideal_lp),
        ideal_lp_recon,
        normalize_for_display(H_gaussian_lp),
        gaussian_lp_recon,
        normalize_for_display(H_ideal_hp),
        ideal_hp_recon,
    ],
    [
        "Ideal low-pass mask",
        "Ideal low-pass reconstruction",
        "Gaussian low-pass mask",
        "Gaussian low-pass reconstruction",
        "Ideal high-pass mask",
        "Ideal high-pass reconstruction",
    ],
    cols=3,
    figsize=(14, 8),
)

show_images(
    [clip_uint8(freq_image), gaussian_lp_recon, gaussian_hp_recon],
    ["Original", "Gaussian low-pass reconstruction", "Gaussian high-pass reconstruction"],
    cols=3,
    figsize=(14, 5),
)

## 9. Reconstruction from Magnitude Only vs. Phase Only

To separate the roles of spectrum magnitude and phase, this cell reconstructs two artificial spectra:

Magnitude-only reconstruction:

$$
F_m(u,v)=|F(u,v)|e^{j0}=|F(u,v)|
$$

Phase-only reconstruction:

$$
F_p(u,v)=1\cdot e^{j\phi(u,v)}
$$

Magnitude stores frequency strength, while phase stores much of the spatial arrangement and structural location information.

In [ ]:
magnitude_only_spectrum = magnitude * np.exp(1j * np.zeros_like(phase))
phase_only_spectrum = np.ones_like(magnitude) * np.exp(1j * phase)

magnitude_only_recon = reconstruct_from_centered_spectrum(magnitude_only_spectrum)
phase_only_recon = reconstruct_from_centered_spectrum(phase_only_spectrum)

show_images(
    [
        clip_uint8(freq_image),
        normalize_for_display(magnitude_only_recon),
        normalize_for_display(phase_only_recon),
    ],
    [
        "Original grayscale image",
        "Reconstruction using magnitude only",
        "Reconstruction using phase only",
    ],
    cols=3,
    figsize=(14, 5),
)

## 10. High-Frequency Energy and Spectral Energy Ratio

The high-frequency energy is computed outside a chosen radius \(D_0\):

$$
E_H=\sum_{(u,v)\in high}|F(u,v)|^2
$$

The spectral energy ratio is

$$
R=\frac{E_H}{E_{total}}
$$

For frequency-domain filtered results, the metric is computed directly on the filtered spectrum, not on a clipped or display-normalized reconstruction. This keeps the quantitative values tied to the actual Fourier coefficients used by the filters.


In [ ]:
def frequency_energy_from_spectrum(spectrum, high_radius=70):
    spectrum = np.asarray(spectrum, dtype=np.complex128)
    power = np.abs(spectrum) ** 2
    D = centered_distance_grid(spectrum.shape)
    high_mask = D > high_radius
    high_energy = float(np.sum(power[high_mask]))
    total_energy = float(np.sum(power))
    ratio = high_energy / total_energy if total_energy > 0 else 0.0
    return high_energy, total_energy, ratio


def frequency_energy_metrics(image, high_radius=70):
    image = np.asarray(image, dtype=np.float64)
    centered = image * checkerboard(image.shape)
    spectrum = fft2(centered)
    return frequency_energy_from_spectrum(spectrum, high_radius=high_radius)


metric_entries = [
    ("Original", frequency_energy_from_spectrum(F, high_radius=70)),
    ("Unsharp masking", frequency_energy_metrics(unsharp, high_radius=70)),
    ("Spatial high-pass enhanced", frequency_energy_metrics(spatial_high_pass_enhanced, high_radius=70)),
    ("Ideal low-pass", frequency_energy_from_spectrum(F * H_ideal_lp, high_radius=70)),
    ("Gaussian low-pass", frequency_energy_from_spectrum(F * H_gaussian_lp, high_radius=70)),
    ("Ideal high-pass", frequency_energy_from_spectrum(F * H_ideal_hp, high_radius=70)),
    ("Gaussian high-pass", frequency_energy_from_spectrum(F * H_gaussian_hp, high_radius=70)),
]

print(f"{'Image':28s} {'High-frequency energy':>24s} {'Total energy':>18s} {'Ratio':>10s}")
print("-" * 86)
for name, (EH, ET, R) in metric_entries:
    print(f"{name:28s} {EH:24.4e} {ET:18.4e} {R:10.6f}")


## 11. Discussion and Interpretation

**Spatial vs. frequency enhancement.** Spatial sharpening works directly with local neighborhoods. Unsharp masking subtracts a smoothed image from the original, so it is simple and interpretable in the image plane. Frequency filtering manipulates explicit frequency bands, so it gives more direct control over low-frequency smooth content and high-frequency edge/noise content.

**Ideal low-pass vs. Gaussian low-pass.** The ideal low-pass filter has an abrupt cutoff in the frequency domain. This sharp discontinuity commonly produces ringing artifacts in the spatial reconstruction, known as the Gibbs phenomenon. The Gaussian low-pass filter decays smoothly, so it usually gives a softer blur with fewer visible rings around sharp transitions.

**Sharpening and noise amplification.** Sharpening increases high-frequency components. Edges are high frequency, but many noise patterns are also high frequency, so sharpening can preserve or emphasize edges while also making noise more visible.

**Frequency components and image meaning.** Low frequencies represent broad illumination and smooth shapes. High frequencies represent edges, fine texture, and noise. Phase carries much of the spatial layout; this is why the phase-only reconstruction often preserves recognizable structure better than the magnitude-only reconstruction.

**Visual sharpness vs. numerical energy.** A higher high-frequency energy ratio usually suggests more sharp detail, but it does not always mean better perceptual quality. Noise and ringing can also increase high-frequency energy. Therefore, numerical frequency metrics should be interpreted together with visual inspection.